# Ascon v2 — Caminho C (CNN2D) — smoke test

Antes de rodar: **Runtime → Change runtime type → GPU (T4 ou melhor, Colab Pro)**.

Ajuste `DRIVE_DATA_DIR` (célula 1) pro caminho no seu Google Drive onde estão
`keyholdout_5class_v2.parquet` (11,8GB) e `v2_folds.json` (30KB).

O smoke test do Caminho C **não** roda `report_eval` — ele testa as 4 variantes
de condicionamento (sum1/raw/log1p/standardized) e imprime o `val_loss` de cada
uma pra escolher a vencedora. Matriz de confusão completa só aparece a partir do
`--mode hpsearch`/`cv` (ver `docs/plano_experimento_v2/07_runbook_execucao.md`).

In [ ]:
# CÉLULA 1 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = "/content/drive/MyDrive/<sua-pasta>/v2_dataset"  # <-- AJUSTAR

In [ ]:
# CÉLULA 2 — Clonar repo + instalar deps
import subprocess, sys, os

REPO_DIR = "/content/ascon"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/K1nginthen0rth/ascon.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "scikit-learn", "xgboost"], check=True)
print("OK.")

In [ ]:
# CÉLULA 3 — Copiar dataset do Drive pro disco local
# (leitura por row-group direto do Drive-mount é lenta/instável; copia uma vez pro disco da VM)
import shutil

os.makedirs(f"{REPO_DIR}/data/processed", exist_ok=True)
for fname in ["keyholdout_5class_v2.parquet", "v2_folds.json"]:
    src = f"{DRIVE_DATA_DIR}/{fname}"
    dst = f"{REPO_DIR}/data/processed/{fname}"
    if not os.path.exists(dst):
        print(f"Copiando {fname} (pode demorar bastante pro parquet de 11,8GB)...")
        shutil.copy(src, dst)
    print(fname, "OK" if os.path.exists(dst) else "FALTANDO", f"{os.path.getsize(dst)/1e9:.2f} GB" if os.path.exists(dst) else "")

In [ ]:
# CÉLULA 4 — Checar GPU + rodar smoke test Caminho C
import torch

assert torch.cuda.is_available(), "Ative GPU em Runtime > Change runtime type"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.chdir(REPO_DIR)
result = subprocess.run([
    sys.executable, "scripts/run_v2_caminhos_bce.py",
    "--path", "C", "--mode", "smoke", "--branch", "controlado",
], cwd=REPO_DIR)
print(f"\nreturncode={result.returncode}")
if result.returncode != 0:
    raise RuntimeError("Smoke test do Caminho C falhou — ver output acima.")

In [ ]:
# CÉLULA 5 — Resultado: val_loss por variante de condicionamento + vencedora
import json

out_dir = f"{REPO_DIR}/reports/v2/caminho_c/controlado"
with open(f"{out_dir}/cnn2d_conditioning_smoke.json", encoding="utf-8") as fh:
    print(json.dumps(json.load(fh), indent=2, ensure_ascii=False))